In [0]:
# =========================================================
# CELL 1: CONFIGURATION
# 07_new_relic_pipeline_health
# =========================================================

import json
import hashlib
import os
from datetime import datetime, timezone

import requests

from pyspark.sql import functions as F
from delta.tables import DeltaTable


spark.conf.set("spark.sql.session.timeZone", "UTC")


# ---------------------------------------------------------
# PROJECT CONFIGURATION
# ---------------------------------------------------------

CATALOG = "training_lab_classic"
SCHEMA = "cold_chain"
VOLUME = "volume1"

NAMESPACE = f"{CATALOG}.{SCHEMA}"

PIPELINE_NAME = "cold_chain_databricks_pipeline"
TEAM_NAME = "cold-chain-capstone"
ENVIRONMENT = "training"


# ---------------------------------------------------------
# TABLE NAMES
# ---------------------------------------------------------

AUDIT_TABLE = f"{NAMESPACE}.pipeline_run_audit"

BRONZE_SENSOR_TABLE = (
    f"{NAMESPACE}.bronze_sensor_events"
)

SILVER_SENSOR_TABLE = (
    f"{NAMESPACE}.silver_sensor_events"
)

QUARANTINE_SENSOR_TABLE = (
    f"{NAMESPACE}.quarantine_sensor_events"
)

ENRICHED_SENSOR_TABLE = (
    f"{NAMESPACE}.silver_sensor_enriched"
)

TEMPERATURE_WINDOWS_TABLE = (
    f"{NAMESPACE}.shipment_temperature_windows"
)

SHIPMENT_HEALTH_TABLE = (
    f"{NAMESPACE}.shipment_health"
)

DELIVERY_PERFORMANCE_TABLE = (
    f"{NAMESPACE}.delivery_performance"
)

TELEMETRY_LOG_TABLE = (
    f"{NAMESPACE}.new_relic_telemetry_log"
)


# ---------------------------------------------------------
# WIDGET HELPERS
# ---------------------------------------------------------

def create_text_widget_if_missing(name, default_value):
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.text(name, default_value)


def create_dropdown_widget_if_missing(
    name,
    default_value,
    allowed_values
):
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.dropdown(
            name,
            default_value,
            allowed_values
        )


create_text_widget_if_missing("run_id", "")

create_dropdown_widget_if_missing(
    "new_relic_region",
    "US",
    ["US", "EU", "JP"]
)

create_dropdown_widget_if_missing(
    "publish_enabled",
    "false",
    ["true", "false"]
)


REQUESTED_RUN_ID = (
    dbutils.widgets.get("run_id").strip()
)

NEW_RELIC_REGION = (
    dbutils.widgets.get("new_relic_region")
    .strip()
    .upper()
)

create_text_widget_if_missing(
    "job_start_time",
    ""
)

JOB_START_TIME_RAW = (
    dbutils.widgets
    .get("job_start_time")
    .strip()
)

print(
    "Job start time raw:",
    JOB_START_TIME_RAW
)

# Publishing now follows the widget.
# Set publish_enabled to true when ready to send.
PUBLISH_ENABLED = (
    dbutils.widgets.get("publish_enabled")
    .strip()
    .lower()
    == "true"
)


assert NEW_RELIC_REGION in {"US", "EU", "JP"}


# Enter the replacement license key locally.
# Never print this value or commit this notebook with the key.
NEW_RELIC_LICENSE_KEY = "d6c95594f27f3574c2187b67ff2a490d624bNRAL"

if PUBLISH_ENABLED:
    assert NEW_RELIC_LICENSE_KEY.strip(), (
        "Configure the replacement New Relic license key "
        "in Cell 1 before publishing."
    )


print("==============================================")
print("NEW RELIC PIPELINE HEALTH")
print("==============================================")
print("Namespace          :", NAMESPACE)
print("Pipeline           :", PIPELINE_NAME)
print("Team               :", TEAM_NAME)
print("Environment        :", ENVIRONMENT)
print("Requested run ID   :", REQUESTED_RUN_ID or "latest completed")
print("New Relic region   :", NEW_RELIC_REGION)
print("Publishing enabled :", PUBLISH_ENABLED)

In [0]:
# Reads the exact parameter names configured in the Job.

parameter_names = [
    "bronze_result",
    "silver_validation_result",
    "silver_reference_result",
    "gold_result",
]

for parameter_name in parameter_names:
    dbutils.widgets.text(parameter_name, "")


def read_task_result(parameter_name):
    raw_value = dbutils.widgets.get(parameter_name).strip()

    assert raw_value and "{{" not in raw_value, (
        f"Missing or unresolved parameter: {parameter_name}. "
        f"Received: {raw_value!r}"
    )

    return raw_value.upper()


actual_bronze_status = read_task_result(
    "bronze_result"
)

actual_silver_validation_status = read_task_result(
    "silver_validation_result"
)

actual_silver_reference_status = read_task_result(
    "silver_reference_result"
)

actual_gold_status = read_task_result(
    "gold_result"
)


silver_results = {
    actual_silver_validation_status,
    actual_silver_reference_status,
}

if silver_results == {"SUCCESS"}:
    actual_silver_status = "SUCCESS"
elif "FAILED" in silver_results:
    actual_silver_status = "FAILED"
elif "UPSTREAM_FAILED" in silver_results:
    actual_silver_status = "UPSTREAM_FAILED"
else:
    actual_silver_status = "UNKNOWN"


print("==============================================")
print("ACTUAL JOB TASK RESULTS")
print("==============================================")
print("Requested Run ID  :", REQUESTED_RUN_ID)
print("Bronze            :", actual_bronze_status)
print("Silver validation :", actual_silver_validation_status)
print("Silver reference  :", actual_silver_reference_status)
print("Combined Silver   :", actual_silver_status)
print("Gold              :", actual_gold_status)

In [0]:
# Publishes a minimal failure event when the pipeline failed
# before its operational audit record could be created.

assert REQUESTED_RUN_ID, (
    "Current Job run_id is required."
)

current_audit_exists = False

if spark.catalog.tableExists(AUDIT_TABLE):
    current_audit_exists = (
        spark.table(AUDIT_TABLE)
        .filter(
            F.col("run_id") == REQUESTED_RUN_ID
        )
        .limit(1)
        .count()
        > 0
    )


if not current_audit_exists:

    task_results = {
        "BRONZE": actual_bronze_status,
        "SILVER_VALIDATION": actual_silver_validation_status,
        "SILVER_REFERENCE": actual_silver_reference_status,
        "GOLD": actual_gold_status,
    }

    # Identify directly failed tasks, not downstream tasks
    # that did not execute because an upstream task failed.
    directly_failed_stages = [
        stage
        for stage, result in task_results.items()
        if result == "FAILED"
    ]

    assert directly_failed_stages, (
        "No audit record exists for this run, and no "
        "direct FAILED task result was supplied. "
        f"Task results: {task_results}. "
        "Do not substitute an older audit record."
    )

    failure_event = {
        "eventType": "ColdChainPipelineRun",
        "team": TEAM_NAME,
        "pipeline": PIPELINE_NAME,
        "environment": ENVIRONMENT,

        "runId": REQUESTED_RUN_ID,
        "databricksRunId": REQUESTED_RUN_ID,

        "status": "FAILED",

        "bronzeStatus": actual_bronze_status,
        "silverValidationStatus": (
            actual_silver_validation_status
        ),
        "silverReferenceStatus": (
            actual_silver_reference_status
        ),
        "silverStatus": actual_silver_status,
        "goldStatus": actual_gold_status,

        "failedStage": directly_failed_stages[0],
        "failedStages": ",".join(
            directly_failed_stages
        ),

        "auditRecordAvailable": False,
        "metricsAvailable": False,
        "statusSource": "databricks_task_results",

        "observedAt": datetime.now(
            timezone.utc
        ).isoformat(),
    }

    print("Run ID       :", REQUESTED_RUN_ID)
    print("Task results :", task_results)
    print("Failed stages:", directly_failed_stages)

    assert PUBLISH_ENABLED, (
        "Failure event prepared, but publication is disabled. "
        "Set publish_enabled=true in the New Relic task."
    )

    # Uses the secret scope/key from your original configuration.
    # Never paste the ingestion key into notebook source.
    assert (
        "NEW_RELIC_LICENSE_KEY" in globals()
        and isinstance(NEW_RELIC_LICENSE_KEY, str)
        and NEW_RELIC_LICENSE_KEY.strip()
    ), (
        "NEW_RELIC_LICENSE_KEY must be configured "
        "before the failure-event cell."
    )

    license_key = NEW_RELIC_LICENSE_KEY.strip()

    endpoint_by_region = {
        "US": (
            "https://insights-collector.newrelic.com"
            "/v1/accounts/events"
        ),
        "EU": (
            "https://insights-collector.eu01.nr-data.net"
            "/v1/accounts/events"
        ),
        "JP": (
            "https://insights-collector.jp.nr-data.net"
            "/v1/accounts/events"
        ),
    }

    response = requests.post(
        endpoint_by_region[NEW_RELIC_REGION],
        headers={
            "Api-Key": license_key,
            "Content-Type": "application/json",
        },
        data=json.dumps(
            [failure_event],
            allow_nan=False
        ),
        timeout=30,
    )

    response.raise_for_status()

    response_body = response.json()

    assert response_body.get("success") is True, (
        "The Event API did not confirm acceptance."
    )

    print("Failure event accepted.")
    print("HTTP status:", response.status_code)

    # Stops this telemetry notebook successfully after reporting.
    # It does not change the failed Bronze task's result.
    dbutils.notebook.exit(
        json.dumps({
            "reportedRunId": REQUESTED_RUN_ID,
            "reportedPipelineStatus": "FAILED",
            "publicationStatus": "ACCEPTED",
        })
    )

else:
    print(
        "Current audit record found. "
        "Continuing with existing telemetry cells."
    )

In [0]:
# =========================================================
# CELL 2: VALIDATE TABLES AND SELECT AUDIT RUN
# =========================================================

required_tables = {
    "pipeline audit": AUDIT_TABLE,
    "Bronze sensors": BRONZE_SENSOR_TABLE,
    "Silver sensors": SILVER_SENSOR_TABLE,
    "sensor quarantine": QUARANTINE_SENSOR_TABLE,
    "enriched sensors": ENRICHED_SENSOR_TABLE,
    "temperature windows": TEMPERATURE_WINDOWS_TABLE,
    "shipment health": SHIPMENT_HEALTH_TABLE,
    "delivery performance": DELIVERY_PERFORMANCE_TABLE
}


print("==============================================")
print("REQUIRED TABLE CHECK")
print("==============================================")


for logical_name, table_name in required_tables.items():
    exists = spark.catalog.tableExists(table_name)

    print(
        f"{logical_name}:",
        "FOUND" if exists else "MISSING"
    )

    assert exists, (
        f"Required table is missing: {table_name}"
    )


audit_df = spark.table(AUDIT_TABLE)


required_audit_columns = {
    "run_id",
    "orchestrator",
    "airflow_dag_id",
    "airflow_run_id",
    "databricks_job_id",
    "databricks_run_id",
    "status",
    "bronze_status",
    "silver_status",
    "gold_status",
    "started_at",
    "completed_at",
    "input_rows",
    "accepted_rows",
    "rejected_rows",
    "rejected_percent",
    "temperature_exception_count",
    "gold_temperature_window_rows",
    "shipment_health_rows",
    "delivery_performance_rows",
    "error_message"
}


missing_audit_columns = (
    required_audit_columns - set(audit_df.columns)
)

assert not missing_audit_columns, (
    "Audit table is missing these columns: "
    f"{sorted(missing_audit_columns)}"
)


assert REQUESTED_RUN_ID, (
    "Current pipeline run_id is required. "
    "Selecting an older completed run is not allowed."
)

selected_audit_df = (
    audit_df
    .filter(
        F.col("run_id") == REQUESTED_RUN_ID
    )
)

selected_rows = selected_audit_df.collect()

assert len(selected_rows) == 1, (
    "Expected exactly one audit record for "
    f"run_id={REQUESTED_RUN_ID}; "
    f"found={len(selected_rows)}. "
    "Initialize/finalize the audit for this Job run first."
)


audit_run = selected_rows[0]
audit_values = audit_run.asDict()

assert audit_values["run_id"] is not None, (
    "The selected audit record has no run_id."
)

RUN_ID = str(audit_values["run_id"])

started_at = audit_values["started_at"]
completed_at = audit_values["completed_at"]


assert started_at is not None, (
    "The selected audit record has no started_at."
)

assert completed_at is not None, (
    "This notebook reports completed runs. "
    "The selected run has no completed_at."
)


# ---------------------------------------------------------
# READ ACTUAL STORED STATUSES
# ---------------------------------------------------------

def audit_status(column_name):
    value = audit_values.get(column_name)

    if value is None or not str(value).strip():
        return "UNKNOWN"

    return str(value).strip().upper()


# Overall status includes operational checks recorded in audit.
pipeline_status = audit_status("status")

# Layer statuses come from actual Job task results.
bronze_status = actual_bronze_status
silver_status = actual_silver_status
gold_status = actual_gold_status

# Never report overall SUCCESS if a tracked task failed.
tracked_results = [
    actual_bronze_status,
    actual_silver_validation_status,
    actual_silver_reference_status,
    actual_gold_status,
]

if "FAILED" in tracked_results:
    pipeline_status = "FAILED"

print("Pipeline          :", pipeline_status)
print("Bronze            :", bronze_status)
print("Silver validation :", actual_silver_validation_status)
print("Silver reference  :", actual_silver_reference_status)
print("Combined Silver   :", silver_status)
print("Gold              :", gold_status)


# Optional failure fields.
# Missing fields are not invented.
failed_stage = audit_values.get("failed_stage")
failed_task = audit_values.get("failed_task")
error_type = audit_values.get("error_type")
error_message = audit_values.get("error_message")
retry_count = audit_values.get("retry_count")


print("==============================================")
print("SELECTED PIPELINE RUN")
print("==============================================")
print("Run ID          :", RUN_ID)
print("Pipeline status :", pipeline_status)
print("Bronze status   :", bronze_status)
print("Silver status   :", silver_status)
print("Gold status     :", gold_status)
print("Started at      :", started_at)
print("Completed at    :", completed_at)
print("Orchestrator    :", audit_values["orchestrator"])
print("Failed stage    :", failed_stage)
print("Failed task     :", failed_task)
print("Error type      :", error_type)
print("Retry count     :", retry_count)

display(selected_audit_df)


In [0]:
print("Selected Run ID:", RUN_ID)
print("Audit input rows:", audit_values.get("input_rows"))

tables_to_check = {
    "Bronze": BRONZE_SENSOR_TABLE,
    "Silver": SILVER_SENSOR_TABLE,
    "Quarantine": QUARANTINE_SENSOR_TABLE,
}

for label, table_name in tables_to_check.items():
    df = spark.table(table_name)

    print(f"\n{label} total rows: {df.count()}")

    if "_run_id" in df.columns:
        display(
            df.groupBy("_run_id")
            .count()
            .orderBy(F.col("count").desc())
        )

        current_count = (
            df.filter(
                F.col("_run_id").cast("string") == RUN_ID
            )
            .count()
        )

        print(f"{label} current-run rows: {current_count}")

    else:
        print(f"{label}: _run_id column missing")

In [0]:
# =========================================================
# CELL 3: DATABRICKS JOB CONTEXT
# =========================================================

def get_context_tag(tag_name):
    try:
        context = (
            dbutils.notebook
            .entry_point
            .getDbutils()
            .notebook()
            .getContext()
        )

        value = context.tags().apply(tag_name)

        if value is None:
            return None

        value = str(value).strip()
        return value or None

    except Exception:
        return None


def optional_string(value):
    if value is None:
        return None

    value = str(value).strip()
    return value or None


# These identify the pipeline run selected from the audit table.
DATABRICKS_JOB_ID = optional_string(
    audit_values["databricks_job_id"]
)

DATABRICKS_RUN_ID = optional_string(
    audit_values["databricks_run_id"]
)


# These identify the notebook/job publishing the telemetry.
MONITORING_JOB_ID = (
    get_context_tag("jobId")
    or get_context_tag("job_id")
)

MONITORING_RUN_ID = (
    get_context_tag("jobRunId")
    or get_context_tag("runId")
    or get_context_tag("job_run_id")
)

DATABRICKS_TASK_KEY = (
    get_context_tag("taskKey")
    or get_context_tag("task_key")
)


print("==============================================")
print("DATABRICKS CONTEXT")
print("==============================================")
print("Pipeline Job ID   :", DATABRICKS_JOB_ID)
print("Pipeline Run ID   :", DATABRICKS_RUN_ID)
print("Monitoring Job ID :", MONITORING_JOB_ID)
print("Monitoring Run ID :", MONITORING_RUN_ID)
print("Monitoring task   :", DATABRICKS_TASK_KEY)

In [0]:
# =========================================================
# CELL 4: CALCULATE PIPELINE METRICS
# =========================================================

def safe_int(value, default_value=0):
    return default_value if value is None else int(value)


def safe_float(value, default_value=None):
    return default_value if value is None else float(value)


# Track whether supporting metrics are run-scoped.
supporting_metric_scopes = {}


def scope_to_selected_run(dataframe, logical_name):
    columns = set(dataframe.columns)

    if "run_id" in columns:
        supporting_metric_scopes[logical_name] = "run_id"

        print(f"{logical_name}: scoped using run_id")

        return dataframe.filter(
            F.col("run_id") == RUN_ID
        )

    if "_processing_time" in columns:
        supporting_metric_scopes[logical_name] = (
            "processing_time"
        )

        print(
            f"{logical_name}: scoped using _processing_time"
        )

        return dataframe.filter(
            (F.col("_processing_time") >= F.lit(started_at))
            & (
                F.col("_processing_time")
                <= F.lit(completed_at)
            )
        )

    supporting_metric_scopes[logical_name] = "full_table"

    print(
        f"WARNING: {logical_name} has no run_id or "
        "_processing_time. Supporting metrics use "
        "the full table, not just the selected run."
    )

    return dataframe


# ---------------------------------------------------------
# READ AND SCOPE SUPPORTING TABLES
# ---------------------------------------------------------

run_bronze_sensor_df = scope_to_selected_run(
    spark.table(BRONZE_SENSOR_TABLE),
    "Bronze sensors"
)

run_silver_sensor_df = scope_to_selected_run(
    spark.table(SILVER_SENSOR_TABLE),
    "Silver sensors"
)

run_quarantine_sensor_df = scope_to_selected_run(
    spark.table(QUARANTINE_SENSOR_TABLE),
    "Quarantine sensors"
)

run_enriched_sensor_df = scope_to_selected_run(
    spark.table(ENRICHED_SENSOR_TABLE),
    "Enriched sensors"
)

run_temperature_windows_df = scope_to_selected_run(
    spark.table(TEMPERATURE_WINDOWS_TABLE),
    "Temperature windows"
)

run_shipment_health_df = scope_to_selected_run(
    spark.table(SHIPMENT_HEALTH_TABLE),
    "Shipment health"
)

run_delivery_performance_df = scope_to_selected_run(
    spark.table(DELIVERY_PERFORMANCE_TABLE),
    "Delivery performance"
)


# ---------------------------------------------------------
# AUDIT COUNTS
# ---------------------------------------------------------

# Current-run ingestion metrics, not cumulative table counts.

def count_current_run(table_name):
    df = spark.table(table_name)

    assert "_run_id" in df.columns, (
        f"_run_id is missing from {table_name}"
    )

    return (
        df.filter(
            F.col("_run_id").cast("string") == str(RUN_ID)
        )
        .count()
    )


input_rows = count_current_run(
    BRONZE_SENSOR_TABLE
)

accepted_rows = count_current_run(
    SILVER_SENSOR_TABLE
)

rejected_rows = count_current_run(
    QUARANTINE_SENSOR_TABLE
)

output_rows = accepted_rows

rejected_percent = (
    rejected_rows / input_rows * 100.0
    if input_rows > 0
    else None
)

bronze_total_rows = spark.table(
    BRONZE_SENSOR_TABLE
).count()

assert accepted_rows + rejected_rows == input_rows, (
    "Current-run reconciliation failed: "
    f"input={input_rows}, "
    f"accepted={accepted_rows}, "
    f"rejected={rejected_rows}"
)

print("Current Run ID     :", RUN_ID)
print("Current-run input  :", input_rows)
print("Current-run accepted:", accepted_rows)
print("Current-run rejected:", rejected_rows)
print("Bronze total stored:", bronze_total_rows)


# Temperature excursions for the selected Job run only.

enriched_sensor = spark.table(
    ENRICHED_SENSOR_TABLE
)

assert "_run_id" in enriched_sensor.columns, (
    "_run_id is missing from the enriched sensor table."
)

temperature_exception_count = (
    enriched_sensor
    .filter(
        F.col("_run_id").cast("string")
        == str(RUN_ID)
    )
    .filter(
        F.col("is_temperature_exception") == True
    )
    .count()
)

print(
    "Current-run temperature excursions:",
    temperature_exception_count
)

gold_temperature_window_rows = safe_int(
    audit_values["gold_temperature_window_rows"]
)

shipment_health_rows = safe_int(
    audit_values["shipment_health_rows"]
)

delivery_performance_rows = safe_int(
    audit_values["delivery_performance_rows"]
)

gold_output_rows = (
    gold_temperature_window_rows
    + shipment_health_rows
    + delivery_performance_rows
)

no_data_condition = input_rows == 0


# ---------------------------------------------------------
# DUPLICATES
# ---------------------------------------------------------

duplicate_rows = None

if "event_id" in run_bronze_sensor_df.columns:
    duplicate_result = (
        run_bronze_sensor_df
        .filter(F.col("event_id").isNotNull())
        .groupBy("event_id")
        .count()
        .filter(F.col("count") > 1)
        .agg(
            F.sum(F.col("count") - F.lit(1))
            .alias("duplicate_rows")
        )
        .first()
    )

    duplicate_rows = int(
        duplicate_result["duplicate_rows"] or 0
    )
else:
    print(
        "WARNING: event_id unavailable. "
        "Duplicate metrics will be omitted."
    )


duplicate_percent = (
    duplicate_rows / input_rows * 100.0
    if duplicate_rows is not None and input_rows > 0
    else None
)


# ---------------------------------------------------------
# BUSINESS HEALTH
# ---------------------------------------------------------

missing_sensor_count = None

if "missing_sensor_status" in run_shipment_health_df.columns:
    missing_sensor_count = (
        run_shipment_health_df
        .filter(
            F.col("missing_sensor_status")
            .isin("MISSING", "STALE")
        )
        .count()
    )


open_overdue_count = None

if (
    "open_overdue_status"
    in run_delivery_performance_df.columns
):
    open_overdue_count = (
        run_delivery_performance_df
        .filter(
            F.col("open_overdue_status") == "OPEN_OVERDUE"
        )
        .count()
    )


# ---------------------------------------------------------
# DURATION AND FRESHNESS
# ---------------------------------------------------------

duration_seconds = float(
    (completed_at - started_at).total_seconds()
)

assert duration_seconds >= 0, (
    "Pipeline duration cannot be negative."
)


latest_event_timestamp = None

if "event_time" in run_enriched_sensor_df.columns:
    latest_event_row = (
        run_enriched_sensor_df
        .select(
            F.expr("try_cast(event_time AS TIMESTAMP)")
            .alias("parsed_event_time")
        )
        .agg(
            F.max("parsed_event_time")
            .alias("latest_event_timestamp")
        )
        .first()
    )

    latest_event_timestamp = (
        latest_event_row["latest_event_timestamp"]
    )


latest_processing_timestamp = None

if "_processing_time" in run_bronze_sensor_df.columns:
    latest_processing_row = (
        run_bronze_sensor_df
        .agg(
            F.max("_processing_time")
            .alias("latest_processing_timestamp")
        )
        .first()
    )

    latest_processing_timestamp = (
        latest_processing_row["latest_processing_timestamp"]
    )


# Preserve your existing definition:
# age of the latest source event at pipeline completion.
freshness_seconds = None

if latest_event_timestamp is not None:
    freshness_seconds = max(
        0.0,
        float(
            (
                completed_at - latest_event_timestamp
            ).total_seconds()
        )
    )


source_to_processing_delay_seconds = None

if {"event_time", "_processing_time"}.issubset(
    set(run_bronze_sensor_df.columns)
):
    delay_row = (
        run_bronze_sensor_df
        .withColumn(
            "parsed_event_time",
            F.expr("try_cast(event_time AS TIMESTAMP)")
        )
        .filter(
            F.col("parsed_event_time").isNotNull()
            & F.col("_processing_time").isNotNull()
        )
        .withColumn(
            "delay_seconds",
            F.unix_timestamp("_processing_time")
            - F.unix_timestamp("parsed_event_time")
        )
        .filter(F.col("delay_seconds") >= 0)
        .agg(
            F.max("delay_seconds")
            .alias("maximum_delay_seconds")
        )
        .first()
    )

    source_to_processing_delay_seconds = safe_float(
        delay_row["maximum_delay_seconds"]
    )


# ---------------------------------------------------------
# REJECTION REASONS
# ---------------------------------------------------------

rejection_reason_counts = {}

if "rejection_reason" in run_quarantine_sensor_df.columns:
    rejection_reason_rows = (
        run_quarantine_sensor_df
        .filter(F.col("rejection_reason").isNotNull())
        .groupBy("rejection_reason")
        .count()
        .orderBy("rejection_reason")
        .collect()
    )

    rejection_reason_counts = {
        str(row["rejection_reason"]): int(row["count"])
        for row in rejection_reason_rows
    }


mandatory_field_reasons = {
    "missing_event_id",
    "missing_shipment_id"
}

invalid_data_type_reasons = {
    "nonnumeric_temperature",
    "invalid_event_time"
}

reference_validation_reasons = {
    "unknown_shipment",
    "unknown_product"
}


mandatory_field_failures = sum(
    count_value
    for reason, count_value in rejection_reason_counts.items()
    if reason in mandatory_field_reasons
)

invalid_data_type_failures = sum(
    count_value
    for reason, count_value in rejection_reason_counts.items()
    if reason in invalid_data_type_reasons
)

reference_validation_failures = sum(
    count_value
    for reason, count_value in rejection_reason_counts.items()
    if reason in reference_validation_reasons
)

# Do not report zero when the audit has no measurement.
schema_validation_failures = audit_values.get(
    "schema_validation_failures"
)


# ---------------------------------------------------------
# VALIDATE BEFORE PUBLICATION
# ---------------------------------------------------------

for metric_name, metric_value in {
    "input_rows": input_rows,
    "accepted_rows": accepted_rows,
    "rejected_rows": rejected_rows,
    "gold_temperature_window_rows": gold_temperature_window_rows,
    "shipment_health_rows": shipment_health_rows,
    "delivery_performance_rows": delivery_performance_rows
}.items():
    assert metric_value >= 0, (
        f"{metric_name} cannot be negative."
    )


if rejected_percent is not None:
    assert 0 <= rejected_percent <= 100

if duplicate_percent is not None:
    assert 0 <= duplicate_percent <= 100


# Keep reconciliation as a warning because duplicate handling
# depends on how your upstream pipeline counts accepted rows.
if (
    accepted_rows + rejected_rows + (duplicate_rows or 0)
    > input_rows
):
    print(
        "WARNING: accepted + rejected + duplicates exceeds "
        "input_rows. Check upstream counting definitions."
    )


print("==============================================")
print("ACTUAL PIPELINE HEALTH METRICS")
print("==============================================")
print("Run ID          :", RUN_ID)
print("Pipeline status :", pipeline_status)
print("Bronze status   :", bronze_status)
print("Silver status   :", silver_status)
print("Gold status     :", gold_status)
print("Duration        :", duration_seconds)
print("Input rows      :", input_rows)
print("Accepted rows   :", accepted_rows)
print("Rejected rows   :", rejected_rows)
print("Duplicate rows  :", duplicate_rows)
print("Gold output rows:", gold_output_rows)

In [0]:
# =========================================================
# CELL 5: BUILD NEW RELIC EVENTS AND PAYLOAD
# =========================================================

def iso_utc(value):
    if value is None:
        return None

    if value.tzinfo is None:
        value = value.replace(tzinfo=timezone.utc)

    return value.astimezone(timezone.utc).isoformat()


def remove_none_values(event):
    return {
        key: value
        for key, value in event.items()
        if value is not None
    }


common_attributes = {
    "team": TEAM_NAME,
    "pipeline": PIPELINE_NAME,
    "environment": ENVIRONMENT,
    "runId": RUN_ID,
    "databricksJobId": DATABRICKS_JOB_ID,
    "databricksRunId": DATABRICKS_RUN_ID
}


# ---------------------------------------------------------
# EVENT 1: PIPELINE HEALTH AND ALL LAYER STATUSES
# ---------------------------------------------------------

pipeline_event = {
    **common_attributes,

    "eventType": "ColdChainPipelineRun",

    "orchestrator": audit_values["orchestrator"],
    "airflowDagId": audit_values["airflow_dag_id"],
    "airflowRunId": audit_values["airflow_run_id"],

    "monitoringJobId": MONITORING_JOB_ID,
    "monitoringRunId": MONITORING_RUN_ID,
    "databricksTaskKey": DATABRICKS_TASK_KEY,

    # Actual audit statuses, not inferred from row counts.
    "status": pipeline_status,
    "bronzeStatus": bronze_status,
    "silverStatus": silver_status,
    "goldStatus": gold_status,

    "failedStage": failed_stage,
    "failedTask": failed_task,
    "errorType": error_type,
    "errorMessage": error_message,
    "retryCount": (
        int(retry_count)
        if retry_count is not None
        else None
    ),

    "startedAt": iso_utc(started_at),
    "completedAt": iso_utc(completed_at),
    "durationSeconds": duration_seconds,

    "inputRows": input_rows,
    "outputRows": output_rows,
    "acceptedRows": accepted_rows,
    "rejectedRows": rejected_rows,
    "duplicateRows": duplicate_rows,
    "rejectedPercent": rejected_percent,
    "duplicatePercent": duplicate_percent,
    "bronzeTotalRows": bronze_total_rows,

    "latestEventTimestamp": iso_utc(
        latest_event_timestamp
    ),
    "latestProcessingTimestamp": iso_utc(
        latest_processing_timestamp
    ),
    "freshnessSeconds": freshness_seconds,
    "freshnessDefinition": (
        "latest_source_event_age_at_run_completion"
    ),
    "sourceToProcessingDelaySeconds": (
        source_to_processing_delay_seconds
    ),

    "noDataCondition": no_data_condition,
    "temperatureExceptionCount": temperature_exception_count,
    "missingSensorCount": missing_sensor_count,
    "openOverdueCount": open_overdue_count,

    "bronzeSensorRows": input_rows,
    "silverAcceptedRows": accepted_rows,
    "silverRejectedRows": rejected_rows,
    "quarantineRows": rejected_rows,

    "goldTemperatureWindowRows": (
        gold_temperature_window_rows
    ),
    "shipmentHealthRows": shipment_health_rows,
    "deliveryPerformanceRows": delivery_performance_rows,
    "goldOutputRows": gold_output_rows,

    # Use a separately recorded KPI status if available.
    # Do not infer it from overall pipeline status.
    "businessKpiGenerationStatus": audit_values.get(
        "business_kpi_generation_status"
    ),

    "goldRefreshTime": (
        iso_utc(completed_at)
        if gold_status.strip().upper() == "SUCCESS"
        else None
    ),

    "supportingMetricsUseFullTable": (
        "full_table" in supporting_metric_scopes.values()
    )
}


# ---------------------------------------------------------
# EVENT 2: DATA QUALITY
# ---------------------------------------------------------

data_quality_event = {
    **common_attributes,

    "eventType": "ColdChainDataQuality",

    "inputRows": input_rows,
    "acceptedRows": accepted_rows,
    "rejectedRows": rejected_rows,
    "duplicateRows": duplicate_rows,
    "rejectedPercent": rejected_percent,
    "duplicatePercent": duplicate_percent,

    "schemaValidationFailures": (
        int(schema_validation_failures)
        if schema_validation_failures is not None
        else None
    ),
    "mandatoryFieldFailures": mandatory_field_failures,
    "referenceValidationFailures": (
        reference_validation_failures
    ),
    "invalidDataTypeFailures": invalid_data_type_failures,

    "duplicateMetricScope": supporting_metric_scopes[
        "Bronze sensors"
    ],
    "rejectionMetricScope": supporting_metric_scopes[
        "Quarantine sensors"
    ]
}


# ---------------------------------------------------------
# EVENT 3: BUSINESS HEALTH
# ---------------------------------------------------------

business_health_event = {
    **common_attributes,

    "eventType": "ColdChainBusinessHealth",

    "temperatureExceptionCount": temperature_exception_count,
    "missingSensorCount": missing_sensor_count,
    "openOverdueCount": open_overdue_count,
    "shipmentHealthRows": shipment_health_rows,
    "deliveryPerformanceRows": delivery_performance_rows,

    "shipmentMetricScope": supporting_metric_scopes[
        "Shipment health"
    ],
    "deliveryMetricScope": supporting_metric_scopes[
        "Delivery performance"
    ],

    "evaluatedAt": iso_utc(completed_at)
}


# ---------------------------------------------------------
# EVENT 4: REJECTION REASONS
# ---------------------------------------------------------

rejection_events = []

for reason, count_value in rejection_reason_counts.items():
    rejection_events.append({
        **common_attributes,

        "eventType": "ColdChainRejectionReason",
        "dataset": "sensor_events",
        "rejectionReason": str(reason),
        "rejectedRows": int(count_value),
        "inputRows": input_rows,

        "metricScope": supporting_metric_scopes[
            "Quarantine sensors"
        ]
    })


# ---------------------------------------------------------
# FINAL PAYLOAD
# ---------------------------------------------------------

new_relic_events = [
    pipeline_event,
    data_quality_event,
    business_health_event
] + rejection_events

new_relic_events = [
    remove_none_values(event)
    for event in new_relic_events
]

assert len(new_relic_events) >= 3


payload_json = json.dumps(
    new_relic_events,
    sort_keys=True,
    separators=(",", ":"),
    allow_nan=False
)


print("==============================================")
print("NEW RELIC STATUS PREVIEW")
print("==============================================")

print(json.dumps({
    "runId": RUN_ID,
    "status": pipeline_status,
    "bronzeStatus": bronze_status,
    "silverStatus": silver_status,
    "goldStatus": gold_status
}, indent=2))

print("Events prepared:", len(new_relic_events))

# Avoid printing error messages or other full event contents.

In [0]:
# =========================================================
# CELL 6: PUBLICATION LOG AND NEW RELIC CONFIGURATION
# =========================================================

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TELEMETRY_LOG_TABLE}
    (
        telemetry_id STRING,
        run_id STRING,
        databricks_job_id STRING,
        databricks_run_id STRING,
        payload_hash STRING,
        event_count BIGINT,
        publish_status STRING,
        http_status BIGINT,
        published_at TIMESTAMP,
        error_message STRING
    )
    USING DELTA
""")


payload_hash = hashlib.sha256(
    payload_json.encode("utf-8")
).hexdigest()

telemetry_id = f"{RUN_ID}:{payload_hash}"


existing_success_count = (
    spark.table(TELEMETRY_LOG_TABLE)
    .filter(
        (F.col("telemetry_id") == telemetry_id)
        & (F.col("publish_status") == "SUCCESS")
    )
    .count()
)

ALREADY_PUBLISHED = existing_success_count > 0


region_endpoints = {
    "US": (
        "https://insights-collector.newrelic.com"
        "/v1/accounts/events"
    ),
    "EU": (
        "https://insights-collector.eu01.nr-data.net"
        "/v1/accounts/events"
    ),
    "JP": (
        "https://insights-collector.jp.nr-data.net"
        "/v1/accounts/events"
    )
}

NEW_RELIC_EVENT_URL = region_endpoints[NEW_RELIC_REGION]


if PUBLISH_ENABLED and not ALREADY_PUBLISHED:
    assert NEW_RELIC_LICENSE_KEY.strip(), (
        "New Relic license key is missing from Cell 1."
    )

print("==============================================")
print("PUBLICATION CONFIGURATION")
print("==============================================")
print("Telemetry ID      :", telemetry_id)
print("Region            :", NEW_RELIC_REGION)
print("Already published :", ALREADY_PUBLISHED)
print("Publishing enabled:", PUBLISH_ENABLED)

In [0]:
# =========================================================
# CELL 7: PUBLISH TO NEW RELIC AND SAVE RESULT
# =========================================================

print("==============================================")
print("NEW RELIC PUBLICATION")
print("==============================================")


# ---------------------------------------------------------
# RECHECK WHETHER THIS PAYLOAD WAS ALREADY PUBLISHED
# ---------------------------------------------------------

ALREADY_PUBLISHED = (
    spark.table(TELEMETRY_LOG_TABLE)
    .filter(
        (F.col("telemetry_id") == telemetry_id)
        & (F.col("publish_status") == "SUCCESS")
    )
    .limit(1)
    .count()
    > 0
)


publication_status = "SKIPPED"
http_status = None
publication_error = None


# ---------------------------------------------------------
# CASE 1: PUBLISHING DISABLED
# ---------------------------------------------------------

if not PUBLISH_ENABLED:
    publication_status = "DISABLED"

    print("Publishing disabled.")
    print("Set the publish_enabled widget to true.")


# ---------------------------------------------------------
# CASE 2: PAYLOAD ALREADY PUBLISHED
# ---------------------------------------------------------

elif ALREADY_PUBLISHED:
    publication_status = "ALREADY_PUBLISHED"

    print(
        "This payload already has a successful "
        "publication record."
    )
    print("Skipping duplicate publication.")


# ---------------------------------------------------------
# CASE 3: SEND EVENTS
# ---------------------------------------------------------

else:
    try:
        assert NEW_RELIC_LICENSE_KEY, (
            "New Relic ingestion key is missing. "
            "Configure the key in Cell 6."
        )

        print("Sending pipeline health events...")

        response = requests.post(
            NEW_RELIC_EVENT_URL,
            headers={
                "Api-Key": NEW_RELIC_LICENSE_KEY,
                "Content-Type": "application/json"
            },
            data=payload_json,
            timeout=30
        )

        http_status = int(response.status_code)

        print("HTTP status:", http_status)

        if not 200 <= http_status < 300:
            raise RuntimeError(
                f"New Relic returned HTTP {http_status}."
            )

        try:
            response_body = response.json()
        except ValueError:
            response_body = None

        if (
            isinstance(response_body, dict)
            and response_body.get("success") is False
        ):
            raise RuntimeError(
                "New Relic returned success=false."
            )

        publication_status = "SUCCESS"

        print("New Relic accepted the telemetry request.")

    except Exception as exc:
        publication_status = "FAILED"

        # Do not print credentials, headers, or request data.
        publication_error = (
            f"{type(exc).__name__}: "
            "Telemetry publication failed. "
            f"HTTP status: {http_status}"
        )

        print(publication_error)


# ---------------------------------------------------------
# PUBLICATION TIMESTAMP
# ---------------------------------------------------------

published_at = datetime.now(
    timezone.utc
).replace(tzinfo=None)


# ---------------------------------------------------------
# SAVE PUBLICATION RESULT
# ---------------------------------------------------------

if publication_status != "ALREADY_PUBLISHED":

    telemetry_log_df = spark.createDataFrame(
        [
            (
                telemetry_id,
                RUN_ID,
                DATABRICKS_JOB_ID,
                DATABRICKS_RUN_ID,
                payload_hash,
                int(len(new_relic_events)),
                publication_status,
                http_status,
                published_at,
                publication_error
            )
        ],
        """
        telemetry_id STRING,
        run_id STRING,
        databricks_job_id STRING,
        databricks_run_id STRING,
        payload_hash STRING,
        event_count BIGINT,
        publish_status STRING,
        http_status BIGINT,
        published_at TIMESTAMP,
        error_message STRING
        """
    )

    telemetry_log_delta = DeltaTable.forName(
        spark,
        TELEMETRY_LOG_TABLE
    )

    # Explicit mappings avoid referencing target-only columns,
    # such as pipeline_status, that are absent from the source.
    log_column_mapping = {
        "telemetry_id": "source.telemetry_id",
        "run_id": "source.run_id",
        "databricks_job_id": "source.databricks_job_id",
        "databricks_run_id": "source.databricks_run_id",
        "payload_hash": "source.payload_hash",
        "event_count": "source.event_count",
        "publish_status": "source.publish_status",
        "http_status": "source.http_status",
        "published_at": "source.published_at",
        "error_message": "source.error_message"
    }

    (
        telemetry_log_delta.alias("target")
        .merge(
            telemetry_log_df.alias("source"),
            "target.telemetry_id = source.telemetry_id"
        )
        .whenMatchedUpdate(
            set=log_column_mapping
        )
        .whenNotMatchedInsert(
            values=log_column_mapping
        )
        .execute()
    )

    print("Publication log saved successfully.")

else:
    print("Existing successful publication record retained.")


# ---------------------------------------------------------
# DISPLAY ALL PIPELINE STATUSES
# ---------------------------------------------------------

print()
print("==============================================")
print("PIPELINE STATUS SUMMARY")
print("==============================================")
print("Run ID          :", RUN_ID)
print("Pipeline status :", pipeline_status)
print("Bronze status   :", bronze_status)
print("Silver status   :", silver_status)
print("Gold status     :", gold_status)
print("Publication     :", publication_status)
print("HTTP status     :", http_status)


# ---------------------------------------------------------
# DISPLAY PUBLICATION LOG
# ---------------------------------------------------------

display(
    spark.table(TELEMETRY_LOG_TABLE)
    .filter(F.col("run_id") == RUN_ID)
    .orderBy(F.col("published_at").desc())
)


# ---------------------------------------------------------
# FAIL CLEARLY IF PUBLICATION FAILED
# ---------------------------------------------------------

assert publication_status in {
    "SUCCESS",
    "ALREADY_PUBLISHED",
    "DISABLED"
}, (
    "New Relic publication failed. "
    "Review the HTTP status and publication log."
)

In [0]:
# =========================================================
# CELL 8: FINAL STATUS SUMMARY
# =========================================================

print("==============================================")
print("PIPELINE AND NEW RELIC SUMMARY")
print("==============================================")

summary = {
    "Pipeline": PIPELINE_NAME,
    "Run ID": RUN_ID,

    "Overall pipeline status": pipeline_status,
    "Bronze status": bronze_status,
    "Silver status": silver_status,
    "Gold status": gold_status,

    "Failed stage": failed_stage,
    "Failed task": failed_task,
    "Error type": error_type,
    "Retry count": retry_count,

    "Databricks Job ID": DATABRICKS_JOB_ID,
    "Databricks Run ID": DATABRICKS_RUN_ID,

    "Duration seconds": duration_seconds,
    "Input rows": input_rows,
    "Accepted rows": accepted_rows,
    "Rejected rows": rejected_rows,
    "Duplicate rows": duplicate_rows,
    "Rejected percent": rejected_percent,
    "Duplicate percent": duplicate_percent,

    "Latest event timestamp": latest_event_timestamp,
    "Latest processing timestamp": latest_processing_timestamp,
    "Freshness seconds": freshness_seconds,
    "Source delay seconds": source_to_processing_delay_seconds,

    "Temperature exceptions": temperature_exception_count,
    "Missing/stale sensors": missing_sensor_count,
    "Open overdue shipments": open_overdue_count,

    "Temperature window rows": gold_temperature_window_rows,
    "Shipment health rows": shipment_health_rows,
    "Delivery performance rows": delivery_performance_rows,
    "Gold output rows": gold_output_rows,

    "Events prepared": len(new_relic_events),
    "Publication status": publication_status,
    "HTTP status": http_status
}

for label, value in summary.items():
    print(f"{label:30}: {value}")


assert publication_status in {
    "SUCCESS",
    "ALREADY_PUBLISHED",
    "DISABLED"
}, (
    "Telemetry publication failed. "
    "Check Cell 7 and the telemetry log."
)


print()
print("==============================================")

if publication_status == "DISABLED":
    print("PREVIEW COMPLETE: no telemetry was sent.")
else:
    print("TELEMETRY PUBLICATION CHECK PASSED.")

print(
    "Actual pipeline status remains:",
    pipeline_status
)

print("==============================================")

In [0]:
display(
    spark.table(
        "training_lab_classic.cold_chain.new_relic_telemetry_log"
    )
    .select(
        "run_id",
        "publish_status",
        "http_status",
        "published_at",
        "error_message"
    )
    .orderBy(F.col("published_at").desc())
    .limit(10)
)